<a href="https://colab.research.google.com/github/m1Febriansyah/244107020199-ML-12/blob/main/JS06ANNPraktikum06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy
!pip install hnswlib
!pip install faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 16.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=36016a21b27642e783db5d618e753a48e8f1851d561c241cedb3c8bbb2c5c0bf
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=6e9bca05291f9737cdcb0136c0988b24ac9e2975982514e92f1913e4805742e2
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.8 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset dari session storage
# -------------------------------
# Menggunakan engine='python' atau on_bad_lines='skip' untuk menangani error formatting pada lirik
try:
    df = pd.read_csv('songs_with_attributes_and_lyrics.csv', on_bad_lines='skip')
except Exception as e:
    print(f"Gagal membaca dengan default engine: {e}")
    print("Mencoba membaca dengan engine='python'...")
    df = pd.read_csv('songs_with_attributes_and_lyrics.csv', engine='python', on_bad_lines='skip')

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF (Memperbaiki error inisialisasi)
# -------------------------------
start = time.time()
# Menggunakan index_factory untuk membuat index IVF100 dengan L2 metric (Flat)
dimension = X_scaled.shape[1]
index_faiss = faiss.index_factory(dimension, "IVF100,Flat", faiss.METRIC_L2)

# Melatih dan menambahkan data
index_faiss.train(X_scaled.astype('float32'))
index_faiss.add(X_scaled.astype('float32'))
index_faiss.nprobe = 10

dist_faiss, idx_faiss = index_faiss.search(X_scaled.astype('float32'), k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Gagal membaca dengan default engine: Error tokenizing data. C error: EOF inside string starting at row 13086
Mencoba membaca dengan engine='python'...
Exact NN done in 1.428 s
Annoy done in 0.550 s
HNSW done in 2.036 s
FAISS IVF done in 0.328 s

Top-5 neighbors for first song:
Exact NN: [    0  3836  2788 10895  9511]
Annoy:    [0, 3836, 2788, 10895, 9511]
HNSW:     [    0  3836  2788 10895  9511]
FAISS:    [    0  3836  2788 10895  9511]


Kode Python tersebut bertujuan untuk melakukan pengujian perbandingan (benchmarking) antara metode Exact Nearest Neighbor berbasis brute-force dan tiga pustaka Approximate Nearest Neighbor (ANN) populer, yaitu Annoy, HNSW, dan FAISS. Pengujian dilakukan dengan memanfaatkan sembilan fitur audio dari dataset Spotify yang telah dinormalisasi menggunakan StandardScaler. Tujuan utamanya adalah mengukur sejauh mana algoritma aproksimasi mampu mempercepat proses pencarian lagu serupa dibandingkan pencarian persis.

Dari sisi mekanisme kerja, metode Exact NN menghitung jarak Euclidean ke seluruh titik data secara menyeluruh untuk menghasilkan akurasi sempurna 100%, tetapi memiliki kompleksitas tinggi pada data berukuran besar. Sebagai alternatif, Annoy membagi ruang fitur menggunakan struktur random projection trees, HNSW membangun graf bernavigasi multi-layer (Hierarchical Navigable Small World) yang sangat efisien, dan FAISS IVF mengelompokkan data ke dalam kluster menggunakan inverted index untuk membatasi area pencarian hanya pada kluster terdekat.

Meskipun struktur kodenya sudah berjalan dengan baik, terdapat beberapa catatan kritis terkait metodologi pengukurannya. Masalah utama terletak pada penggabungan waktu pembuatan indeks (build time) dan waktu pencarian (query time) dalam satu variabel eksekusi. Pada sistem produksi nyata, pembuatan indeks dilakukan secara offline, sedangkan pencarian dilakukan berulang kali secara real-time. Selain itu, penggunaan loop Python pada Annoy membuat perbandingan waktu menjadi kurang adil akibat overhead interpreter, serta belum adanya pengukuran metrik Recall@k untuk menguji seberapa akurat hasil aproksimasi tersebut dibandingkan hasil persisnya.

Secara keseluruhan, HNSW umumnya memberikan keseimbangan terbaik antara kecepatan pencarian dan tingkat recall yang tinggi pada dataset skala kecil hingga menengah. FAISS IVF menjadi pilihan paling tepat saat menangani dataset berukuran raksasa karena efisiensi pemanfaatan memorinya, sedangkan Annoy unggul dalam kemudahan distribusi model berbasis file. Untuk mendapatkan laporan evaluasi yang lebih valid, penulisan kode sebaiknya disempurnakan dengan memisahkan pencatatan waktu indexing dan querying, mengeksekusi pencarian secara batch, mengonversi tipe data ke float32, serta mengukur nilai Recall@k.